In [10]:
API_KEY_ALPACA_PAPER = 'PKMLW0XE86NIOEV6I63V' #paper new account johri.rajat+1@gmail.com
SECRET_KEY_ALPACA_PAPER = 'aaIllPGt8RQodGb55DFeJKQoPjQuvPsYQWhrY1B7' #paper new account johri.rajat+1@gmail.com

# from alpaca.data.historical import StockHistoricalDataClient
# from alpaca.data.requests import StockLatestQuoteRequest
# from alpaca.data.requests import StockBarsRequest
# #from alpaca.data.timeframe import TimeFrame
# from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo  # Available in Python 3.9+
import pytz
import os
import pandas as pd
import numpy as np
import logging
import time
import requests
#from datetime import time

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from math import sqrt

symbol = "SPY"

In [11]:
# --- 3. Define Timezone ---
eastern = ZoneInfo('US/Eastern')  # Define Eastern Time Zone

# --- 4. Define API Endpoint and Headers ---
BASE_URL = "https://data.alpaca.markets/v2/stocks"
headers = {
    "APCA-API-KEY-ID": API_KEY_ALPACA_PAPER,
    "APCA-API-SECRET-KEY": SECRET_KEY_ALPACA_PAPER
}

# --- 5. Define Request Parameters ---

timeframe = "5min"
start_date_et = datetime(2000, 1, 1, tzinfo=eastern)  # January 1, 2024, ET
end_date_et = datetime.now(eastern) - timedelta(days=1)  # Yesterday, ET
end_date_et = end_date_et.replace(hour=0, minute=0, second=0, microsecond=0)  # Midnight ET

# Convert ET datetime to UTC for API request
start_date_utc = start_date_et.astimezone(ZoneInfo('UTC'))
end_date_utc = end_date_et.astimezone(ZoneInfo('UTC'))

# Format dates in RFC-3339 with 'Z' to indicate UTC
start_date_str = start_date_utc.isoformat().replace('+00:00', 'Z')
end_date_str = end_date_utc.isoformat().replace('+00:00', 'Z')

limit = 10000  # Maximum allowed per request

# --- 6. Initialize Variables for Pagination ---
all_bars = []
next_page_token = None

# --- 7. Function to Fetch Data ---
def fetch_bars(symbol, timeframe, start, end, limit, page_token=None):
    """
    Fetch historical bars for a single symbol.

    Parameters:
        symbol (str): Stock symbol (e.g., "QQQ")
        timeframe (str): Timeframe for aggregation (e.g., "5Min")
        start (str): RFC-3339 formatted start datetime in UTC
        end (str): RFC-3339 formatted end datetime in UTC
        limit (int): Maximum number of data points per request (1-10000)
        page_token (str, optional): Token for pagination

    Returns:
        tuple: (data (dict), headers (dict))
    """
    url = f"{BASE_URL}/{symbol}/bars"
    params = {
        "timeframe": timeframe,
        "start": start,
        "end": end,
        "limit": limit,
        "adjustment": "raw",  # No adjustment
        "asof": "-",  # Skip symbol mapping
        "feed": "sip",  # Standard US exchanges
        "currency": "USD",
        "sort": "asc"  # Ascending order
    }
    if page_token:
        params["page_token"] = page_token

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        logger.error(f"Failed to fetch data: {response.status_code} - {response.text}")
        response.raise_for_status()

    return response.json(), response.headers

# --- 8. Function to Monitor Rate Limits ---
def monitor_rate_limits(response_headers):
    """
    Monitor and log the rate limit information from response headers.

    Parameters:
        response_headers (dict): Headers from the API response
    """
    rate_limit = response_headers.get('X-RateLimit-Limit')
    rate_remaining = response_headers.get('X-RateLimit-Remaining')
    rate_reset = response_headers.get('X-RateLimit-Reset')

    logger.info(f"Rate Limit: {rate_limit}, Remaining: {rate_remaining}, Reset at UNIX epoch: {rate_reset}")

    if rate_remaining is not None and rate_reset is not None:
        rate_remaining = int(rate_remaining)
        rate_reset = int(rate_reset)
        if rate_remaining < 10:
            # Calculate sleep time until reset
            reset_time = datetime.fromtimestamp(rate_reset, tz=ZoneInfo('UTC'))
            current_time = datetime.now(tz=ZoneInfo('UTC'))
            sleep_seconds = (reset_time - current_time).total_seconds() + 1  # Adding 1 second buffer
            if sleep_seconds > 0:
                logger.warning(f"Approaching rate limit. Sleeping for {sleep_seconds} seconds until reset.")
                time.sleep(sleep_seconds)

# --- 9. Fetch Data with Pagination ---
while True:
    logger.info(f"Fetching data from {start_date_str} to {end_date_str} with limit {limit} and page_token {next_page_token}")
    try:
        data, response_headers = fetch_bars(symbol, timeframe, start_date_str, end_date_str, limit, next_page_token)
    except Exception as e:
        logger.error(f"An error occurred during API request: {e}")
        break

    # Monitor rate limits
    monitor_rate_limits(response_headers)

    bars = data.get("bars", [])
    all_bars.extend(bars)
    logger.info(f"Retrieved {len(bars)} bars. Total bars collected: {len(all_bars)}")

    next_page_token = data.get("next_page_token")

    if not next_page_token:
        logger.info("No more pages to fetch.")
        break

    # To respect rate limits, introduce a short delay
    time.sleep(0.5)  # Sleep for half a second



# --- 10. Process and Convert Data --- IMPROVED CODE FOR NO WARNING 20241117
if all_bars:
    df = pd.DataFrame(all_bars).copy()  # Ensure df is a copy

    # Check if DataFrame is not empty
    if not df.empty:
        # Convert 't' (timestamp) to datetime and set timezone to UTC
        #df.loc[:, 't'] = pd.to_datetime(df['t'], utc=True)
        df['t'] = pd.to_datetime(df['t'], utc=True, errors='coerce')

        # Convert timestamps to Eastern Time
        df.loc[:, 'timestamp'] = df['t'].dt.tz_convert(eastern)


        # Convert 'currency' to string if present
        if 'currency' in df.columns:
            df.loc[:, 'currency'] = df['currency'].astype(str)

        # Rename columns for clarity
        rename_mapping = {
            't': 'timestamp_utc',
            'o': 'open',
            'h': 'high',
            'l': 'low',
            'c': 'close',
            'v': 'volume',
            'n': 'trade_count',
            'vw': 'vwap'
        }
        df.rename(columns=rename_mapping, inplace=True)

        # Select and reorder columns, handling the presence of optional fields
        columns_order = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'currency']
        existing_columns = [col for col in columns_order if col in df.columns]
        df = df.loc[:, existing_columns]

        # Sort the DataFrame by timestamp in Eastern Time
        df.sort_values('timestamp', inplace=True)

        # Reset index
        df.reset_index(drop=True, inplace=True)

        # --- 11. Export Data to CSV ---
        csv_filename = f"{symbol}_5min_data_{start_date_et.date()}_to_{end_date_et.date()}.csv"
        df.to_csv(csv_filename, index=False)
        logger.info(f"Data successfully exported to {csv_filename}")

        # --- 12. Display a Sample of the Data ---
        print(df.head())
    else:
        logger.warning("Fetched data is empty.")
else:
    logger.warning("No data was fetched.")

INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token None
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830691
INFO:__main__:Retrieved 2153 bars. Total bars collected: 2153
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTQ1MzMwNzcwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830692
INFO:__main__:Retrieved 2235 bars. Total bars collected: 4388
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTQ1NDY4OTIwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830693
INFO:__main__:Retrieved 2208 bars. Total bars collected: 6596
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTQ1NjMyNjYwMDAwMDAwMDAwMA==
INFO:__main_

INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830715
INFO:__main__:Retrieved 2532 bars. Total bars collected: 69996
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTUwMTUyMTkwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830715
INFO:__main__:Retrieved 2506 bars. Total bars collected: 72502
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTUwMzMxNjgwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830716
INFO:__main__:Retrieved 2453 bars. Total bars collected: 74955
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTUwNTEzMzkwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830717
INFO:__main__:Retrieved 2571 bars. Total b

INFO:__main__:Retrieved 2314 bars. Total bars collected: 136918
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU0ODc3MjgwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830739
INFO:__main__:Retrieved 2347 bars. Total bars collected: 139265
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU1MDIyNzgwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830740
INFO:__main__:Retrieved 2385 bars. Total bars collected: 141650
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU1MTk3NTkwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830741
INFO:__main__:Retrieved 2354 bars. Total bars collected: 144004
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 

INFO:__main__:Retrieved 2073 bars. Total bars collected: 201210
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU5Mjk0OTAwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830764
INFO:__main__:Retrieved 2168 bars. Total bars collected: 203378
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU5NDM4NjAwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830765
INFO:__main__:Retrieved 2243 bars. Total bars collected: 205621
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTU5NTk0OTMwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830766
INFO:__main__:Retrieved 2296 bars. Total bars collected: 207917
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 

INFO:__main__:Retrieved 2360 bars. Total bars collected: 265084
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTYzNjExNjMwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830788
INFO:__main__:Retrieved 2333 bars. Total bars collected: 267417
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTYzNzc0NTA2MDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830789
INFO:__main__:Retrieved 2169 bars. Total bars collected: 269586
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTYzOTE2MzcwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830790
INFO:__main__:Retrieved 2256 bars. Total bars collected: 271842
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 

INFO:__main__:Retrieved 2226 bars. Total bars collected: 326877
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTY3ODEwMTMwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830811
INFO:__main__:Retrieved 2154 bars. Total bars collected: 329031
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTY3OTQyMjUwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830812
INFO:__main__:Retrieved 2247 bars. Total bars collected: 331278
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTY4MDgwNDkwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830813
INFO:__main__:Retrieved 2309 bars. Total bars collected: 333587
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 

INFO:__main__:Retrieved 2343 bars. Total bars collected: 392823
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTcyMjM2OTYwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830835
INFO:__main__:Retrieved 2234 bars. Total bars collected: 395057
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTcyMzc0MTIwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830836
INFO:__main__:Retrieved 2421 bars. Total bars collected: 397478
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 2025-06-12T04:00:00Z with limit 10000 and page_token U1BZfE18MTcyNTQ2NDEwMDAwMDAwMDAwMA==
INFO:__main__:Rate Limit: 10000, Remaining: 9999, Reset at UNIX epoch: 1749830837
INFO:__main__:Retrieved 2377 bars. Total bars collected: 399855
INFO:__main__:Fetching data from 2000-01-01T05:00:00Z to 

                  timestamp    open    high     low   close  volume  \
0 2015-12-31 19:00:00-05:00  204.05  204.08  204.05  204.08     500   
1 2015-12-31 19:05:00-05:00  204.01  204.05  204.01  204.05     750   
2 2015-12-31 19:10:00-05:00  204.09  204.09  204.01  204.01     350   
3 2015-12-31 19:45:00-05:00  204.04  204.04  204.04  204.04     300   
4 2015-12-31 19:50:00-05:00  204.02  204.07  204.02  204.07     650   

   trade_count        vwap  
0            6  204.057500  
1            3  204.024430  
2            2  204.032857  
3            2  204.040000  
4            3  204.031538  


### UTC FIXED

In [14]:
# --- 10. Process and Convert Data --- (Keep Alpaca's default UTC timestamps)
if all_bars:
    df = pd.DataFrame(all_bars).copy()  # Ensure df is a copy

    if not df.empty:
        # Convert 't' (timestamp) to datetime and set timezone to UTC (keep as is, no tz conversion)
        df['t'] = pd.to_datetime(df['t'], utc=True, errors='coerce')

        # Rename columns for clarity
        rename_mapping = {
            't': 'timestamp_utc',  # Keep as UTC timestamp, no conversion
            'o': 'open',
            'h': 'high',
            'l': 'low',
            'c': 'close',
            'v': 'volume',
            'n': 'trade_count',
            'vw': 'vwap'
        }
        df.rename(columns=rename_mapping, inplace=True)

        # Convert 'currency' to string if present
        if 'currency' in df.columns:
            df['currency'] = df['currency'].astype(str)

        # Select and reorder columns, handling the presence of optional fields
        columns_order = ['timestamp_utc', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'currency']
        existing_columns = [col for col in columns_order if col in df.columns]
        df = df.loc[:, existing_columns]

        # Sort the DataFrame by the UTC timestamp
        df.sort_values('timestamp_utc', inplace=True)

        # Reset index
        df.reset_index(drop=True, inplace=True)

        # --- 11. Export Data to CSV ---
        csv_filename = f"{symbol}_5min_data_{start_date_et.date()}_to_{end_date_et.date()}.csv"
        df.to_csv(csv_filename, index=False)
        logger.info(f"Data successfully exported to {csv_filename}")

        # --- 12. Display a Sample of the Data ---
        print(df.head())
    else:
        logger.warning("Fetched data is empty.")
else:
    logger.warning("No data was fetched.")


INFO:__main__:Data successfully exported to SPY_5min_data_2000-01-01_to_2025-06-12.csv


              timestamp_utc    open    high     low   close  volume  \
0 2016-01-01 00:00:00+00:00  204.05  204.08  204.05  204.08     500   
1 2016-01-01 00:05:00+00:00  204.01  204.05  204.01  204.05     750   
2 2016-01-01 00:10:00+00:00  204.09  204.09  204.01  204.01     350   
3 2016-01-01 00:45:00+00:00  204.04  204.04  204.04  204.04     300   
4 2016-01-01 00:50:00+00:00  204.02  204.07  204.02  204.07     650   

   trade_count        vwap  
0            6  204.057500  
1            3  204.024430  
2            2  204.032857  
3            2  204.040000  
4            3  204.031538  


In [19]:
from datetime import time as time_class
# --- 3. Ensure 'timestamp' is datetime and in Eastern Time ---
df['timestamp'] = pd.to_datetime(df['timestamp_utc'])

# Check if 'timestamp' is timezone-aware
if df['timestamp'].dt.tz is None:
    # Localize to Eastern Time
    eastern = ZoneInfo('US/Eastern')
    df['timestamp'] = df['timestamp'].dt.tz_localize(eastern)
else:
    # Convert to Eastern Time if it's in a different timezone
    df['timestamp'] = df['timestamp'].dt.tz_convert('US/Eastern')
    
#csv3_filename = "SPY_1MIN.csv"
#df.to_csv(csv3_filename, index=False)

In [21]:
print(df)

                   timestamp_utc      open    high     low     close  volume  \
0      2016-01-01 00:00:00+00:00  204.0500  204.08  204.05  204.0800     500   
1      2016-01-01 00:05:00+00:00  204.0100  204.05  204.01  204.0500     750   
2      2016-01-01 00:10:00+00:00  204.0900  204.09  204.01  204.0100     350   
3      2016-01-01 00:45:00+00:00  204.0400  204.04  204.04  204.0400     300   
4      2016-01-01 00:50:00+00:00  204.0200  204.07  204.02  204.0700     650   
...                          ...       ...     ...     ...       ...     ...   
433787 2025-06-11 23:35:00+00:00  600.6000  600.94  600.60  600.8300   15273   
433788 2025-06-11 23:40:00+00:00  600.8300  601.01  600.83  600.8301    3968   
433789 2025-06-11 23:45:00+00:00  600.8300  600.83  600.56  600.5600    4170   
433790 2025-06-11 23:50:00+00:00  600.5000  600.58  600.48  600.5800    4563   
433791 2025-06-11 23:55:00+00:00  600.5601  600.60  600.45  600.4895   13274   

        trade_count        vwap        

In [13]:
from datetime import time as time_class
# --- 3. Ensure 'timestamp' is datetime and in Eastern Time ---
df2=df.copy()
df2['timestamp'] = pd.to_datetime(df2['timestamp'])

# Check if 'timestamp' is timezone-aware
if df2['timestamp'].dt.tz is None:
    # Localize to Eastern Time
    eastern = ZoneInfo('US/Eastern')
    df2['timestamp'] = df2['timestamp'].dt.tz_localize(eastern)
else:
    # Convert to Eastern Time if it's in a different timezone
    df2['timestamp'] = df2['timestamp'].dt.tz_convert('US/Eastern')
    
print(df2)

                       timestamp      open    high     low     close  volume  \
0      2015-12-31 19:00:00-05:00  204.0500  204.08  204.05  204.0800     500   
1      2015-12-31 19:05:00-05:00  204.0100  204.05  204.01  204.0500     750   
2      2015-12-31 19:10:00-05:00  204.0900  204.09  204.01  204.0100     350   
3      2015-12-31 19:45:00-05:00  204.0400  204.04  204.04  204.0400     300   
4      2015-12-31 19:50:00-05:00  204.0200  204.07  204.02  204.0700     650   
...                          ...       ...     ...     ...       ...     ...   
433787 2025-06-11 19:35:00-04:00  600.6000  600.94  600.60  600.8300   15273   
433788 2025-06-11 19:40:00-04:00  600.8300  601.01  600.83  600.8301    3968   
433789 2025-06-11 19:45:00-04:00  600.8300  600.83  600.56  600.5600    4170   
433790 2025-06-11 19:50:00-04:00  600.5000  600.58  600.48  600.5800    4563   
433791 2025-06-11 19:55:00-04:00  600.5601  600.60  600.45  600.4895   13274   

        trade_count        vwap  
0    